# Is my model any good?
**Machine Learning · Week 1 (Learning from examples) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w1d5.html

Accuracy is one number. Today: the **confusion matrix** and the scores built from it,
on a dataset where one class is rare.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=2000, n_features=2, n_informative=2, n_redundant=0,
                           weights=[0.95, 0.05], class_sep=1.0, flip_y=0, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(f"{len(y)} examples, {y.sum()} positive ({y.mean():.0%})")
print(f"test set: {len(y_test)} examples, {y_test.sum()} positive")

## The lazy baseline
A "model" that always predicts the majority class. Every real model should beat it.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score

lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = lazy.predict(X_test)
print(f"accuracy          {accuracy_score(y_test, pred):.1%}")
print(f"recall (found)    {recall_score(y_test, pred):.1%}")
print(f"balanced accuracy {balanced_accuracy_score(y_test, pred):.1%}")

## A real model, and its confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
pred = knn.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["negative", "positive"],
                                        cmap="Purples", colorbar=False)
plt.title("kNN (k = 5) on the test set")
plt.grid(False)
plt.show()
print(classification_report(y_test, pred, target_names=["negative", "positive"], digits=3))

## Moving the threshold
kNN gives a score: the fraction of the 5 neighbors that are positive. Calling an example
positive at a **lower** threshold finds more positives (recall up) but raises more false alarms (precision down).

In [ ]:
from sklearn.metrics import precision_score, f1_score

score = knn.predict_proba(X_test)[:, 1]
print("threshold  precision  recall   F1    balanced acc")
for t in [0.2, 0.4, 0.6, 0.8, 1.0]:
    p = (score >= t).astype(int)
    print(f"   {t:.1f}      {precision_score(y_test, p, zero_division=0):6.1%}  {recall_score(y_test, p):6.1%}"
          f"  {f1_score(y_test, p):5.1%}   {balanced_accuracy_score(y_test, p):6.1%}")

## Your turn: the scores by hand
Get the four counts from the confusion matrix and compute precision, recall, F1 and balanced
accuracy **yourself**. Then check them against scikit-learn's numbers above.

In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print("TN, FP, FN, TP =", tn, fp, fn, tp)
precision = None   # TODO: of the examples called positive, the fraction that really are
recall = None      # TODO: of the real positives, the fraction we found
print("precision:", precision, " recall:", recall)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
precision = tp / (tp + fp)
recall = tp / (tp + fn)                    # also called sensitivity, or true positive rate
specificity = tn / (tn + fp)               # the recall of the negative class
f1 = 2 * precision * recall / (precision + recall)
balanced = (recall + specificity) / 2
print(f"precision {precision:.3f}   recall {recall:.3f}   F1 {f1:.3f}   balanced accuracy {balanced:.3f}")
print(f"scikit-learn says: F1 {f1_score(y_test, pred):.3f}, balanced accuracy {balanced_accuracy_score(y_test, pred):.3f}")